# Text Classification — Yelp Review Sentiment Analysis

Binary sentiment classification on Yelp reviews (`class`: 1 = negative, 2 = positive).

**Approach:**
1. Load & preprocess the data
2. Exploratory data analysis
3. Baseline: TF-IDF + Logistic Regression
4. Transformer: fine-tuned DistilBERT
5. Evaluation, comparison & inference demo

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch.utils.data import Dataset
from transformers import (DistilBertTokenizer, DistilBertForSequenceClassification,
                          Trainer, TrainingArguments)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

import gc
sns.set_style("whitegrid")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## 1. Load the Data

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
train.columns = ['class', 'text']
test.columns = ['class', 'text']
print(f"Train: {train.shape}, Test: {test.shape}")
train.head()

Train: (54404, 2), Test: (37999, 2)


   class                                               text
0      2  Been going to Dr. Goldberg for over 10 years. ...
1      1  I don't know what Dr. Goldberg was like before...
2      1  I'm writing this review to give you a heads up...
3      2  All the food is great here. But the best thing...
4      1  Wing sauce is like water. Pretty much a lot of...

## 2. Preprocessing

Remap labels to `0 = negative`, `1 = positive`. The original row-by-row loop
(`train['class'][i] = 0`) never actually writes back — pandas Copy-on-Write makes
chained assignment a no-op — and it's slow. Use a vectorized `.map()` instead.

In [3]:
label_map = {1: 0, 2: 1}   # 0 = negative, 1 = positive
train['class'] = train['class'].map(label_map)
test['class'] = test['class'].map(label_map)

# Clean: drop missing values, unescape literal \n sequences
train = train.dropna(subset=['text', 'class']).reset_index(drop=True)
test = test.dropna(subset=['text', 'class']).reset_index(drop=True)
train['text'] = train['text'].astype(str).str.replace(r'\\n', ' ', regex=True)
test['text'] = test['text'].astype(str).str.replace(r'\\n', ' ', regex=True)

print(train['class'].value_counts().sort_index())
train.head()

class
0    27604
1    26800
Name: count, dtype: int64


   class                                               text
0      1  Been going to Dr. Goldberg for over 10 years. ...
1      0  I don't know what Dr. Goldberg was like before...
2      0  I'm writing this review to give you a heads up...
3      1  All the food is great here. But the best thing...
4      0  Wing sauce is like water. Pretty much a lot of...

## 3. Exploratory Data Analysis

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.countplot(x='class', data=train, ax=axes[0])
axes[0].set_title('Class Distribution (Train)')
axes[0].set_xticklabels(['Negative (0)', 'Positive (1)'])

train['length'] = train['text'].str.len()
sns.histplot(data=train, x='length', hue='class', bins=80, kde=True, ax=axes[1])
axes[1].set_title('Review Length by Class')
axes[1].set_xlim(0, 2000)

plt.tight_layout()
plt.show()

<Figure size 1300x400 with 2 Axes>

## 4. Sampling for Experiments

The full training set is large, so we work with a stratified subsample.
Increase `TRAIN_SAMPLE` / `TEST_SAMPLE` (or set them to `None` to use everything) for a full run.

In [5]:
TRAIN_SAMPLE = 20_000   # set to None to use the full training set
TEST_SAMPLE = 5_000

def stratified_sample(df, n, seed=42):
    if n is None or n >= len(df):
        return df.reset_index(drop=True)
    return (df.groupby('class', group_keys=False)
              .apply(lambda g: g.sample(n // 2, random_state=seed))
              .reset_index(drop=True))

train_df = stratified_sample(train, TRAIN_SAMPLE)
test_df = stratified_sample(test, TEST_SAMPLE)
print(f"Using {len(train_df)} train / {len(test_df)} test samples")
print(train_df['class'].value_counts().sort_index().to_dict())

Using 20000 train / 5000 test samples
{0: 10000, 1: 10000}


## 5. Baseline — TF-IDF + Logistic Regression

A strong, fast classical baseline before reaching for transformers.

In [6]:
tfidf = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2),
                        stop_words='english', min_df=2)
X_train = tfidf.fit_transform(train_df['text'])
X_test = tfidf.transform(test_df['text'])
y_train, y_test = train_df['class'].values, test_df['class'].values

lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
print(f"F1 (macro): {f1_score(y_test, lr_preds, average='macro'):.4f}")
print(classification_report(y_test, lr_preds, target_names=['negative', 'positive']))

Accuracy: 0.9070
F1 (macro): 0.9070
              precision    recall  f1-score   support

    negative       0.91      0.91      0.91      2500
    positive       0.91      0.91      0.91      2500

    accuracy                           0.91      5000
   macro avg       0.91      0.91      0.91      5000
weighted avg       0.91      0.91      0.91      5000



In [7]:
cm = confusion_matrix(y_test, lr_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negative', 'positive'], yticklabels=['negative', 'positive'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('TF-IDF + Logistic Regression — Confusion Matrix')
plt.show()

<Figure size 500x400 with 2 Axes>

## 6. DistilBERT Fine-Tuning

Fine-tune `distilbert-base-uncased` for sequence classification.
DistilBERT is ~40% smaller and ~60% faster than BERT while retaining ~97% of its
performance — a good fit for CPU / small-GPU training.

In [8]:
# Free baseline memory before loading the transformer
del X_train, X_test
gc.collect()

MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)

class YelpDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.encodings = tokenizer(list(texts), truncation=True, padding=True,
                                   max_length=max_len)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

BERT_TRAIN = 2_000   # subsample for a quick CPU-friendly fine-tune
BERT_TEST = 500

bert_train_df = stratified_sample(train_df, BERT_TRAIN)
bert_test_df = stratified_sample(test_df, BERT_TEST)

train_ds = YelpDataset(bert_train_df['text'], bert_train_df['class'], tokenizer)
test_ds = YelpDataset(bert_test_df['text'], bert_test_df['class'], tokenizer)
print(f"BERT train: {len(train_ds)}, BERT test: {len(test_ds)}")

BERT train: 2000, BERT test: 500


In [9]:
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds),
            'f1': f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='no',
    logging_steps=20,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()


[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TrainOutput(global_step=250, training_loss=0.39506481647491454, metrics={'train_runtime': 996.243, 'train_samples_per_second': 2.008, 'train_steps_per_second': 0.251, 'total_flos': 66233699328000.0, 'train_loss': 0.39506481647491454, 'epoch': 1.0})

## 7. Evaluation — DistilBERT

In [10]:
bert_out = trainer.predict(test_ds)
bert_preds = np.argmax(bert_out.predictions, axis=-1)
bert_labels = bert_out.label_ids

print(f"Accuracy: {accuracy_score(bert_labels, bert_preds):.4f}")
print(f"F1 (macro): {f1_score(bert_labels, bert_preds, average='macro'):.4f}")
print(classification_report(bert_labels, bert_preds, target_names=['negative', 'positive']))

cm = confusion_matrix(bert_labels, bert_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['negative', 'positive'], yticklabels=['negative', 'positive'])
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.title('DistilBERT — Confusion Matrix')
plt.show()

Accuracy: 0.9080
F1 (macro): 0.9080
              precision    recall  f1-score   support

    negative       0.90      0.91      0.91       250
    positive       0.91      0.90      0.91       250

    accuracy                           0.91       500
   macro avg       0.91      0.91      0.91       500
weighted avg       0.91      0.91      0.91       500



<Figure size 500x400 with 2 Axes>

## 8. Model Comparison

In [11]:
results = pd.DataFrame({
    'Model': ['TF-IDF + Logistic Regression', 'DistilBERT (fine-tuned)'],
    'Accuracy': [accuracy_score(y_test, lr_preds),
                 accuracy_score(bert_labels, bert_preds)],
    'F1 (macro)': [f1_score(y_test, lr_preds, average='macro'),
                   f1_score(bert_labels, bert_preds, average='macro')],
})
results

                          Model  Accuracy  F1 (macro)
0  TF-IDF + Logistic Regression     0.907    0.907000
1       DistilBERT (fine-tuned)     0.908    0.907999

## 9. Inference — Try Your Own Review

In [12]:
def predict_sentiment(texts, model, tokenizer, device):
    model.eval()
    enc = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors='pt')
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    preds = torch.argmax(logits, dim=-1).cpu().numpy()
    probs = torch.softmax(logits, dim=-1).cpu().numpy()
    return [('positive' if p == 1 else 'negative', round(float(probs[i][p]), 3))
            for i, p in enumerate(preds)]

samples = [
    "The food was absolutely amazing and the staff were so friendly!",
    "Terrible service. We waited an hour and the order was wrong.",
    "Best pizza in town — highly recommend!",
    "I will never come back here again. Awful experience.",
]
for text, (label, conf) in zip(samples, predict_sentiment(samples, model, tokenizer, device)):
    print(f"[{label:>8} | {conf:.3f}] {text}")

[positive | 0.971] The food was absolutely amazing and the staff were so friendly!
[negative | 0.967] Terrible service. We waited an hour and the order was wrong.
[positive | 0.948] Best pizza in town — highly recommend!
[negative | 0.961] I will never come back here again. Awful experience.


## 10. Conclusion

- **TF-IDF + Logistic Regression** is a fast, strong baseline for binary sentiment —
  minutes to train, competitive accuracy.
- **DistilBERT fine-tuning** captures context (negation, phrasing) better and typically
  edges out the linear baseline, at the cost of much longer training time.
- Next steps: train on the full dataset, more epochs with early stopping,
  hyperparameter search, and try `roberta-base` for comparison.